# Navier–Stokes regularity and blowup candidates

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/navier_stokes_blowup)

The Clay Millennium Problem asks whether smooth 3D incompressible flows stay smooth forever. A pseudo-spectral solver on the torus runs the Taylor–Green vortex for Euler ($\nu = 0$) and two viscosities as one batch of FFTs, tracking the maximum vorticity, the enstrophy and the Beale–Kato–Majda integral $\int\|\omega\|_\infty dt$ that must diverge at any blowup. Vortex stretching makes the Euler vorticity grow until the grid can no longer resolve it; viscosity caps it. This is an illustration, not a proof.

**Vector form**

$$\partial_t\mathbf u + (\mathbf u\cdot\nabla)\mathbf u = -\nabla p + \nu\nabla^2\mathbf u,\qquad \nabla\cdot\mathbf u = 0,\qquad \frac{dE}{dt} = -2\nu\,\Omega,\qquad T^* < \infty \iff \int_0^{T^*}\!\|\boldsymbol\omega\|_\infty\,dt = \infty$$

**Index (tensor) form**

$$\partial_t\hat u_{bi} = P_{ij}\,\widehat{(\mathbf u\times\boldsymbol\omega)}_{bj} - \nu_b\,k_lk_l\,\hat u_{bi},\qquad P_{ij} = \delta_{ij} - \frac{k_ik_j}{k_lk_l},\qquad \hat\omega_{bi} = i\,\varepsilon_{ijl}\,k_j\,\hat u_{bl}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\hat u_{bi}(\mathbf k)$ | `rfftn(u0, dim=(-3, -2, -1)) * mask` | (B, 3, N, N, N/2+1) complex | velocity Fourier coefficients of every run $b$ (one viscosity each) on the torus $[0, 2\pi)^3$ |
| $\hat\omega_{bi} = i\varepsilon_{ijl}k_j\hat u_{bl}$ | `1j * einsum('ijl,jxyz,blxyz->bixyz', EPS, k, u_hat)` | (B, 3, N, N, M) | curl with the Levi-Civita tensor; repeated $j, l$ ⇒ summed |
| $(\mathbf u\times\boldsymbol\omega)_i$ | `einsum('ijl,bjxyz,blxyz->bixyz', EPS, u, w)` | (B, 3, N, N, N) | the nonlinear term in rotational form, evaluated in real space (pseudo-spectral) |
| $P_{ij}\hat f_j$ | `f_hat - k * (k_dot_f / k2)` | (B, 3, N, N, M) | Leray projection: removes the gradient part, so the pressure is never computed and $k_i\hat u_i = 0$ |
| $\text{2/3 rule}$ | `mask = \|k_i\| < N/3` | (N, N, M) bool | drops the modes that the quadratic term would alias; $k_\max = N/3$ |
| $E,\ \Omega$ | `0.5 * einsum('bixyz,bixyz->b', u, u) / N**3` | (B,) | kinetic energy $\frac12\langle/\mathbf u/^2\rangle$ and enstrophy $\frac12\langle/\boldsymbol\omega/^2\rangle$ |
| $\int_0^t\\vert \omega\\vert _\infty dt$ | `bkm_integral(t, wmax)` | (T, B) | Beale–Kato–Majda criterion: a smooth solution can only break down at $T^*$ if this diverges there |
| $\delta(t)$ | `analyticity_strip(spec, 3, k_max - 1)` | (B,) | fit $E(k) \propto k^{-3}e^{-2\delta k}$; a real singularity needs $\delta \to 0$, and the grid resolves the flow only while $\delta\,k_\max \gtrsim 1$ |

The Clay problem (Fefferman's official description) asks for one of four statements: global smooth solutions on $\mathbb R^3$ (A) or on the torus $\mathbb T^3$ (B), or breakdown on $\mathbb R^3$ (C) or on $\mathbb T^3$ (D), for $\nu > 0$. The repository openai/NavierStokesAndEuler on GitHub states that it contains Lean 4 machine-checked proofs, with accompanying papers, of breakdown for forced Navier–Stokes on $\mathbb R^3$ and $\mathbb T^3$ (alternatives C and D) and of finite-time blowup for the unforced Euler equation from smooth, compactly supported data. These are recent claims that still await assessment by the mathematical community and the Clay Institute; this page does not treat the prize problem as settled. For Euler, finite-time singularity of smooth solutions was supported by the Luo–Hou (2014) axisymmetric computation with a boundary and later by computer-assisted proofs (Chen–Hou), and Elgindi (2021) proved blowup for $C^{1,\alpha}$ data. The simulation here only illustrates the quantities those works control. The Taylor–Green vortex $\mathbf u_0 = (\sin x\cos y\cos z, -\cos x\sin y\cos z, 0)$ is the classic test (Brachet et al. 1983): vortex stretching steepens it into thin sheets, the enstrophy rises, and the energy spectrum flattens. The tail of the spectrum decays like $e^{-2\delta k}$; a blowup would need the analyticity strip $\delta(t)$ to reach zero in finite time, while Brachet found it shrinking only exponentially. Once $\delta\,k_\max < 1$ the Euler run is under-resolved (shaded): the truncated system still conserves energy exactly, but its further growth of $\|\omega\|_\infty$ is a grid artefact, not evidence of a singularity. With viscosity the energy obeys $dE/dt = -2\nu\Omega$ (checked to round-off) and enstrophy peaks and decays. The 2D Taylor–Green vortex is an exact solution $e^{-2\nu t}$ and the 3D one has a known initial acceleration from $p_0 = \frac1{16}(\cos 2x + \cos 2y)(\cos 2z + 2)$: both are tests. Try a larger $N$ to push the resolved time later.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/navier_stokes_blowup/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import analyticity_strip, bkm_integral


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = 16 if fast_mode() else params["n"] // 2 * 2  # even grid
    nu_list = [0.0, params["nu"], 2 * params["nu"]]  # Euler and two viscous runs, solved as one batch
    nu = torch.tensor(nu_list, dtype=torch.float64, device=dev)
    t_max = 1.0 if fast_mode() else params["t_max"]

    k, k2, mask = tensors.wavenumbers(n, dev)  # (3, N, N, M), (N, N, M), (N, N, M)
    u0 = tensors.taylor_green(n, dev)[None].repeat(len(nu_list), 1, 1, 1, 1)  # (B, 3, N, N, N)
    dt = 1.6 / n  # CFL: |u| ≤ 1, Δx = 2π/N
    out = solve.simulate(u0, nu, k, k2, mask, t_max, dt, n_save=8 if fast_mode() else 60)
    t, e, z, wmax = out["t"], out["energy"], out["enstrophy"], out["max_vorticity"]
    bkm = bkm_integral(t, wmax)  # (T, B)

    k_max = n // 3  # largest wavenumber kept by the 2/3 rule
    delta = torch.stack([analyticity_strip(s, 3, k_max - 1) for s in out["spectra"]])  # (S, B)
    resolved = delta[:, 0] * k_max >= 1.0  # a common criterion: the strip is wider than one shortest wave
    t_lost = out["t_save"][~resolved][0].item() if (~resolved).any() else None

    de_dt = (e[2:] - e[:-2]) / (t[2:] - t[:-2])[:, None]
    err = (de_dt + 2 * nu * z[1:-1]).abs().max(dim=0).values / (2 * nu * z).max(dim=0).values.clamp_min(1e-30)
    print(f"Taylor–Green vortex on {n}³ points (2/3 dealiasing, k_max = {k_max}), RK4 with dt = {dt:.4f}")
    for b, v in enumerate(nu_list):
        name = "Euler (ν = 0)" if v == 0 else f"ν = {v:.4f} (Re = {1 / v:.0f})"
        z_peak = z[:, b].max().item()
        t_peak = t[z[:, b].argmax()].item()
        print(
            f"  {name:<22} max Ω = {z_peak:7.3f} at t = {t_peak:4.2f},  ‖ω‖_∞(end) = {wmax[-1, b]:7.3f},  "
            f"BKM ∫‖ω‖_∞ dt = {bkm[-1, b]:6.2f},  E(end)/E₀ = {e[-1, b] / e[0, b]:.4f}"
        )
    print(
        f"energy budget: Euler |E/E₀ − 1| = {abs(e[-1, 0] / e[0, 0] - 1):.1e};  "
        f"viscous max|dE/dt + 2νΩ| / max 2νΩ = {err[1]:.1e}, {err[2]:.1e}"
    )
    ts = out["t_save"]
    picks = [int(torch.searchsorted(ts, torch.tensor(float(c), dtype=ts.dtype))) for c in range(1, int(t_max) + 1)]
    strip = ", ".join(f"δ({ts[i]:.1f}) = {delta[i, 0]:.3f}" for i in picks if i < len(ts))
    print(f"analyticity strip of the Euler run (shrinking, but no sign of reaching 0 while resolved): {strip}")
    viscous_ok = (delta[1:, 1:] * k_max).min().item()
    print(f"resolution of the viscous runs: min δ·k_max = {viscous_ok:.2f} (≥ 1: resolved throughout)")
    if t_lost is not None:
        print(f"Euler run under-resolved after t ≈ {t_lost:.2f} (δ·k_max < 1): later growth is a grid artefact")
    pvd = plot.export_vtk(out, t_lost, out_dir)  # interactive 3D results: the |ω| iso-surface over time
    size = sum(p.stat().st_size for p in pvd.parent.iterdir()) / 1e6
    print(f"interactive results: {pvd.relative_to(out_dir)} and its layers ({size:.1f} MB)")
    return plot.animate(out, nu_list, bkm, delta, k_max, t_lost, out_dir / "navier_stokes_blowup.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/navier_stokes_blowup/equation.py
r"""Physics → tensor translation: incompressible Navier–Stokes in Fourier space (pseudo-spectral, rotational form).

    ∂_t û_i = P_ij(k) [u × ω]^_j − ν |k|² û_i,      P_ij = δ_ij − k_i k_j / |k|²,      ω̂_i = i ε_ijl k_j û_l

b indexes the batch of runs (one viscosity each), i, j, l the three components; repeated j, l are summed. Writing
(u·∇)u = ω × u + ∇(½|u|²) moves the gradient into the pressure, and the projector P removes every gradient, so the
pressure never has to be computed: P keeps the velocity divergence-free (k_i P_ij = 0).

Diagnostics in real space: E = ½⟨|u|²⟩, enstrophy Ω = ½⟨|ω|²⟩ (dE/dt = −2νΩ), ‖ω‖_∞ and the Beale–Kato–Majda
integral ∫₀^t ‖ω‖_∞ dt′, which must diverge if a smooth solution blows up at time t (BKM 1984).
"""

import torch

# Levi-Civita symbol ε_ijl: (3, 3, 3)
EPS = torch.zeros(3, 3, 3, dtype=torch.float64)
for _i, _j, _l in [(0, 1, 2), (1, 2, 0), (2, 0, 1)]:
    EPS[_i, _j, _l], EPS[_i, _l, _j] = 1.0, -1.0


def curl_hat(u_hat: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """ω̂_bi = i ε_ijl k_j û_bl.   u_hat (B, 3, N, N, M) complex, k (3, N, N, M) → (B, 3, N, N, M)"""
    eps = EPS.to(device=k.device, dtype=u_hat.dtype)
    return 1j * torch.einsum("ijl,jxyz,blxyz->bixyz", eps, k.to(u_hat.dtype), u_hat)


def cross(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """(a × b)_i = ε_ijl a_j b_l pointwise.   (B, 3, N, N, N) each"""
    return torch.einsum("ijl,bjxyz,blxyz->bixyz", EPS.to(device=a.device, dtype=a.dtype), a, b)


def project(f_hat: torch.Tensor, k: torch.Tensor, k2: torch.Tensor) -> torch.Tensor:
    """P_ij f̂_j = f̂_i − k_i (k_j f̂_j) / |k|²: removes the gradient (pressure) part.   (B, 3, N, N, M)"""
    kc = k.to(f_hat.dtype)
    k_dot_f = torch.einsum("jxyz,bjxyz->bxyz", kc, f_hat)  # k_j f̂_j
    return f_hat - kc[None] * (k_dot_f / torch.where(k2 == 0, 1.0, k2))[:, None]


def rhs(u_hat: torch.Tensor, nu: torch.Tensor, k: torch.Tensor, k2: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """∂_t û = P[(u × ω)^] − ν|k|² û with the 2/3 dealiasing mask.   nu (B,) one viscosity per run."""
    n = k.shape[1]
    shape = (n, n, n)
    u = torch.fft.irfftn(u_hat, s=shape, dim=(-3, -2, -1))  # (B, 3, N, N, N)
    w = torch.fft.irfftn(curl_hat(u_hat, k), s=shape, dim=(-3, -2, -1))
    nonlinear = torch.fft.rfftn(cross(u, w), dim=(-3, -2, -1)) * mask  # (u × ω)^, aliased modes dropped
    return project(nonlinear, k, k2) - nu[:, None, None, None, None] * k2 * u_hat


def energy(u: torch.Tensor) -> torch.Tensor:
    """E_b = ½⟨u_i u_i⟩ over the box.   (B, 3, N, N, N) → (B,)"""
    return 0.5 * torch.einsum("bixyz,bixyz->b", u, u) / u[0, 0].numel()


def enstrophy(w: torch.Tensor) -> torch.Tensor:
    """Ω_b = ½⟨ω_i ω_i⟩.   (B, 3, N, N, N) → (B,)"""
    return energy(w)


def max_vorticity(w: torch.Tensor) -> torch.Tensor:
    """‖ω‖_∞ = max over the grid of |ω|.   (B, 3, N, N, N) → (B,)"""
    return torch.sqrt(torch.einsum("bixyz,bixyz->bxyz", w, w)).flatten(1).max(dim=1).values


def bkm_integral(t: torch.Tensor, wmax: torch.Tensor) -> torch.Tensor:
    """Beale–Kato–Majda ∫₀^t ‖ω‖_∞ dt′ by the trapezoid rule.   t (T,), wmax (T, B) → (T, B)"""
    steps = 0.5 * (wmax[1:] + wmax[:-1]) * (t[1:] - t[:-1])[:, None]
    return torch.cat([torch.zeros_like(wmax[:1]), torch.cumsum(steps, dim=0)])


def spectrum(u_hat: torch.Tensor, k: torch.Tensor, k2: torch.Tensor) -> torch.Tensor:
    """Shell-averaged energy spectrum E(κ) = Σ_{κ−½ ≤ |k| < κ+½} ½|û_k|² / N⁶.   (B, 3, N, N, M) → (B, K)"""
    n = k.shape[1]
    weight = torch.where((k[2] == 0) | (k[2] == n // 2), 1.0, 2.0)  # rfft stores each ±k_z pair once
    density = 0.5 * weight * (u_hat.abs() ** 2).sum(dim=1) / n**6  # (B, N, N, M)
    shell = torch.sqrt(k2).round().long().flatten()  # (N·N·M,)
    out = torch.zeros(u_hat.shape[0], int(shell.max()) + 1, dtype=density.dtype, device=density.device)
    return out.index_add_(1, shell, density.flatten(1))


def analyticity_strip(spec: torch.Tensor, k_lo: int, k_hi: int) -> torch.Tensor:
    """δ from a fit E(κ) ∝ κ^{-3} e^{−2δκ} on κ ∈ [k_lo, k_hi]: the distance from the real axis to the nearest
    complex singularity. A real singularity at time T* would need δ(T*) = 0.   (B, K) → (B,)"""
    kappa = torch.arange(k_lo, k_hi + 1, dtype=spec.dtype, device=spec.device)  # (K',)
    s = spec[:, k_lo : k_hi + 1]
    w = (s > 1e-26 * spec.max(dim=1, keepdim=True).values).to(spec.dtype)  # fit only shells above round-off
    y = torch.log(s.clamp_min(1e-300)) + 3 * torch.log(kappa)  # (B, K')
    n = w.sum(dim=1)
    k_mean = torch.einsum("bk,k->b", w, kappa) / n.clamp_min(1)
    y_mean = torch.einsum("bk,bk->b", w, y) / n.clamp_min(1)
    kc = kappa[None] - k_mean[:, None]
    slope = torch.einsum("bk,bk,bk->b", w, kc, y - y_mean[:, None]) / torch.einsum("bk,bk,bk->b", w, kc, kc)
    return torch.where(n >= 3, -slope / 2, torch.inf)  # < 3 shells excited: no measurable tail yet (δ = ∞)


def sample_periodic(f: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Trilinear interpolation of a periodic vector field at arbitrary points.
    f (F, 3, N, N, N) on the grid x_j = 2πj/N, x (F, S, 3) positions → (F, S, 3). The 8 corner weights are products
    w = Π_i (1 − |x_i/Δ − c_i|) over the cell corners c, gathered with wrap-around indices."""
    n = f.shape[-1]
    g = x * (n / (2 * torch.pi))  # grid units
    base = torch.floor(g)
    frac = g - base
    base = base.long()
    frames = torch.arange(f.shape[0], device=f.device)[:, None]  # (F, 1)
    out = torch.zeros_like(x)
    for corner in range(8):
        c = torch.tensor([(corner >> 2) & 1, (corner >> 1) & 1, corner & 1], device=x.device)  # (3,)
        idx = (base + c) % n  # (F, S, 3)
        w = torch.where(c.bool(), frac, 1 - frac).prod(dim=-1)  # (F, S)
        val = f[frames, :, idx[..., 0], idx[..., 1], idx[..., 2]]  # (F, S, 3)
        out = out + w[..., None] * val
    return out


def vortex_lines(w: torch.Tensor, seeds: torch.Tensor, n_steps: int, h: float) -> torch.Tensor:
    """Field lines of ω (dx/ds = ω/|ω|), traced both ways from every seed with midpoint (RK2) steps of length h.
    w (F, 3, N, N, N), seeds (F, S, 3) → (F, S, 2·n_steps + 1, 3), wrapped into [0, 2π)³; all frames and seeds at
    once."""

    def direction(x: torch.Tensor) -> torch.Tensor:
        v = sample_periodic(w, x % (2 * torch.pi))
        return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)

    branches = []
    for sign in (1.0, -1.0):
        x, path = seeds, []
        for _ in range(n_steps):
            mid = x + 0.5 * sign * h * direction(x)
            x = x + sign * h * direction(mid)
            path.append(x)
        branches.append(torch.stack(path, dim=2))  # (F, S, n_steps, 3)
    line = torch.cat([branches[1].flip(2), seeds[:, :, None], branches[0]], dim=2)
    return line % (2 * torch.pi)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/navier_stokes_blowup/tensors.py
"""Build the concrete tensors: the periodic grid, the wavevectors with the 2/3 dealiasing mask, and the Taylor–Green
initial velocity fields. Everything lives on the torus T³ = [0, 2π)³ with N points per side."""

import math

import torch


def grid(n: int, device: torch.device) -> torch.Tensor:
    """x_j = 2π j / N, j = 0 … N−1: (N,)"""
    return torch.arange(n, dtype=torch.float64, device=device) * (2 * math.pi / n)


def wavenumbers(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Integer wavevectors of the real FFT: k (3, N, N, M) with M = N/2 + 1, |k|² (N, N, M), and the 2/3-rule mask
    (N, N, M) that zeroes every mode with some |k_i| > N/3 (removes aliasing from the quadratic term)."""
    k_full = torch.fft.fftfreq(n, 1 / n).to(device=device, dtype=torch.float64)  # 0, 1, …, N/2−1, −N/2, …, −1
    k_half = torch.fft.rfftfreq(n, 1 / n).to(device=device, dtype=torch.float64)  # 0 … N/2
    kx, ky, kz = torch.meshgrid(k_full, k_full, k_half, indexing="ij")
    k = torch.stack([kx, ky, kz])  # (3, N, N, M)
    k2 = torch.einsum("ixyz,ixyz->xyz", k, k)  # |k|², repeated i summed
    cut = n / 3
    mask = (kx.abs() < cut) & (ky.abs() < cut) & (kz.abs() < cut)  # (N, N, M) bool
    return k, k2, mask


def taylor_green(n: int, device: torch.device) -> torch.Tensor:
    """3D Taylor–Green vortex u = (sin x cos y cos z, −cos x sin y cos z, 0): (3, N, N, N). E₀ = 1/8."""
    x = grid(n, device)
    X, Y, Z = torch.meshgrid(x, x, x, indexing="ij")
    return torch.stack(
        [torch.sin(X) * torch.cos(Y) * torch.cos(Z), -torch.cos(X) * torch.sin(Y) * torch.cos(Z), torch.zeros_like(X)]
    )


def taylor_green_2d(n: int, device: torch.device) -> torch.Tensor:
    """2D Taylor–Green vortex u = (sin x cos y, −cos x sin y, 0), independent of z: an exact decaying solution."""
    x = grid(n, device)
    X, Y, _ = torch.meshgrid(x, x, x, indexing="ij")
    return torch.stack([torch.sin(X) * torch.cos(Y), -torch.cos(X) * torch.sin(Y), torch.zeros_like(X)])


def surface_nets(field: torch.Tensor, level: float, periodic: bool = False) -> tuple[torch.Tensor, torch.Tensor]:
    """Iso-surface {field = level} by naive surface nets, for all cells at once.

    field (Nx, Ny, Nz) on integer grid points. Every cell (cube of 8 neighbouring points) whose corners straddle the
    level gets one vertex: the mean of the points where the level crosses its 12 edges (linear interpolation along
    each edge). Every grid edge that crosses the level gets a quad joining the 4 cells around it, split into two
    triangles oriented away from the inside (field > level). periodic=True wraps all three directions (a torus).
    Returns vertices (V, 3) in grid units and triangles (F, 3) of vertex indices."""
    dims = torch.tensor(field.shape, device=field.device)
    cells = tuple(int(n) if periodic else int(n) - 1 for n in field.shape)

    def at(offset):  # field at cell corner (i, j, k) + offset, for every cell
        f = torch.roll(field, shifts=tuple(-int(o) for o in offset), dims=(0, 1, 2)) if any(offset) else field
        return f[: cells[0], : cells[1], : cells[2]]

    corners = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]
    values = {c: at(c) - level for c in corners}
    edges = [(p, q) for p in corners for q in corners if p < q and sum(abs(x - y) for x, y in zip(p, q)) == 1]
    total = torch.zeros(*cells, 3, dtype=field.dtype, device=field.device)
    count = torch.zeros(cells, dtype=field.dtype, device=field.device)
    for p, q in edges:  # the 12 edges of a cell (a loop over edge types, not over cells)
        a, b = values[p], values[q]
        cross = (a > 0) != (b > 0)
        t = torch.where(cross, a / (a - b).masked_fill(~cross, 1.0), torch.zeros_like(a))
        point = torch.tensor(p, dtype=field.dtype, device=field.device) + t[..., None] * torch.tensor(
            [qq - pp for pp, qq in zip(p, q)], dtype=field.dtype, device=field.device)  # fmt: skip
        total += point * cross[..., None]
        count += cross
    active = count > 0
    grid = torch.stack(torch.meshgrid(*[torch.arange(n, device=field.device) for n in cells], indexing="ij"), -1)
    verts = (grid + total / count.clamp(min=1)[..., None])[active]  # (V, 3)
    index = torch.full(cells, -1, dtype=torch.long, device=field.device)
    index[active] = torch.arange(int(active.sum()), device=field.device)

    inside = field > level
    faces = []
    for axis in range(3):  # grid edges along each axis
        b_ax, c_ax = [d for d in range(3) if d != axis]
        step = [0, 0, 0]
        step[axis] = 1
        nxt = torch.roll(inside, shifts=-1, dims=axis)
        crossing = inside != nxt
        if not periodic:  # the edge and all four cells around it must lie inside the box
            valid = torch.zeros_like(crossing)
            sl = [slice(None)] * 3
            sl[axis] = slice(0, int(dims[axis]) - 1)
            sl[b_ax] = slice(1, int(dims[b_ax]) - 1)
            sl[c_ax] = slice(1, int(dims[c_ax]) - 1)
            valid[tuple(sl)] = True
            crossing &= valid
        p = crossing.nonzero()  # (E, 3) edge start points
        flip = ~inside[p[:, 0], p[:, 1], p[:, 2]] ^ (axis == 1)  # (axis, b, c) is left-handed for the y axis

        around = []  # vertex index of the cells (p − db e_b − dc e_c) sharing this edge
        for db, dc in ((0, 0), (1, 0), (1, 1), (0, 1)):
            q = p.clone()
            q[:, b_ax] -= db
            q[:, c_ax] -= dc
            q %= torch.tensor(cells, device=field.device)
            around.append(index[q[:, 0], q[:, 1], q[:, 2]])
        v00, v10, v11, v01 = around
        quad = torch.stack([v00, v10, v11, v01], 1)
        quad = torch.where(flip[:, None], quad.flip(1), quad)
        faces.append(torch.cat([quad[:, [0, 1, 2]], quad[:, [0, 2, 3]]]))
    faces = torch.cat(faces)
    return verts, faces[(faces >= 0).all(1)]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/navier_stokes_blowup/solve.py
"""RK4 time stepping of the whole batch of runs (one viscosity each) in Fourier space, with diagnostics every step."""

import torch

from problems._common.progress import steps

from .equation import curl_hat, energy, enstrophy, max_vorticity, rhs, spectrum


def simulate(
    u0: torch.Tensor,
    nu: torch.Tensor,
    k: torch.Tensor,
    k2: torch.Tensor,
    mask: torch.Tensor,
    t_max: float,
    dt: float,
    n_save: int,
) -> dict:
    """u0 (B, 3, N, N, N) real, nu (B,). Returns the time series E, Ω, ‖ω‖_∞ (T, B), n_save snapshots of max_z |ω|
    (S, B, N, N) with their spectra (S, B, K), the Euler run's full ω field at the same times (S, 3, N, N, N), and the
    final û."""
    n = u0.shape[-1]
    shape = (n, n, n)
    u_hat = torch.fft.rfftn(u0, dim=(-3, -2, -1)) * mask
    n_steps = max(1, round(t_max / dt))
    dt = t_max / n_steps
    save_at = {round(s * n_steps / max(1, n_save - 1)) for s in range(n_save)}
    t, e, z, wm, slices, spectra, t_save, w_first = [], [], [], [], [], [], [], []

    def record(step: int) -> None:
        u = torch.fft.irfftn(u_hat, s=shape, dim=(-3, -2, -1))
        w = torch.fft.irfftn(curl_hat(u_hat, k), s=shape, dim=(-3, -2, -1))
        t.append(step * dt)
        e.append(energy(u))
        z.append(enstrophy(w))
        wm.append(max_vorticity(w))
        if step in save_at:
            t_save.append(step * dt)
            slices.append(torch.sqrt((w**2).sum(dim=1)).amax(dim=-1))  # max over z of |ω|: (B, N, N)
            spectra.append(spectrum(u_hat, k, k2))
            w_first.append(w[0].float())  # first run of the batch (Euler), for the 3D view

    record(0)
    for step in steps(n_steps, "RK4 steps"):
        k1 = rhs(u_hat, nu, k, k2, mask)
        k2_ = rhs(u_hat + 0.5 * dt * k1, nu, k, k2, mask)
        k3 = rhs(u_hat + 0.5 * dt * k2_, nu, k, k2, mask)
        k4 = rhs(u_hat + dt * k3, nu, k, k2, mask)
        u_hat = u_hat + dt / 6 * (k1 + 2 * k2_ + 2 * k3 + k4)
        record(step + 1)

    return {
        "t": torch.tensor(t, dtype=torch.float64),
        "energy": torch.stack(e),
        "enstrophy": torch.stack(z),
        "max_vorticity": torch.stack(wm),
        "t_save": torch.tensor(t_save, dtype=torch.float64),
        "slices": torch.stack(slices),
        "spectra": torch.stack(spectra),
        "w_first": torch.stack(w_first),
        "u_hat": u_hat,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/navier_stokes_blowup/plot.py
"""The Euler vorticity in 3D (strong-|ω| region and vortex lines), max_z |ω| for Euler and a viscous run, the growth of
‖ω‖_∞ and Ω, and E(k)."""

import base64
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from PIL import Image

from problems._common.plotting import ACCENT, export_plot_data, frame_indices
from problems._common.progress import report

from .equation import sample_periodic, vortex_lines


def vorticity_3d(
    w: torch.Tensor, n_seeds: int = 20, n_points: int = 2500
) -> tuple[list, np.ndarray, np.ndarray, np.ndarray]:
    """For every saved frame: the grid points with |ω| ≥ ½ the initial max (a fixed level), and vortex lines seeded
    among the strongest points. w (F, 3, N, N, N) → per-frame point clouds (positions, |ω|), lines (F, S, L, 3),
    |ω| along the lines (F, S, L) and max |ω| (F,)."""
    n = w.shape[-1]
    mag = w.norm(dim=1)  # (F, N, N, N)
    peak = mag.flatten(1).amax(dim=1)  # (F,)
    x = torch.arange(n, dtype=w.dtype, device=w.device) * (2 * np.pi / n)
    X = torch.stack(torch.meshgrid(x, x, x, indexing="ij"), dim=-1).reshape(-1, 3)  # (N³, 3)
    flat = (mag / peak[:, None, None, None]).flatten(1)  # (F, N³)
    clouds = []
    for f in range(w.shape[0]):
        keep = torch.nonzero(mag[f].flatten() >= 0.5 * peak[0]).squeeze(1)  # fixed level: ½ the initial max |ω|
        keep = keep[:: max(1, len(keep) // n_points)]
        clouds.append((X[keep].cpu().numpy(), (flat[f, keep] * peak[f]).cpu().numpy()))  # colour: absolute |ω|
    top = flat.topk(20 * n_seeds, dim=1).indices[:, ::20]  # (F, S): spread through the 20·S strongest points
    lines = vortex_lines(w, X[top], n_steps=40, h=0.8 * 2 * np.pi / n)  # (F, S, L, 3)
    f, s_, l_ = lines.shape[:3]
    along = sample_periodic(w, lines.reshape(f, s_ * l_, 3)).norm(dim=-1).reshape(f, s_, l_)  # absolute |ω|
    return clouds, lines.cpu().numpy(), along.cpu().numpy(), peak.cpu().numpy()


def line_segments(lines: np.ndarray, along: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Consecutive-point segments of every line (S, L, 3) with the mean |ω| of their ends, dropping the segments where
    a periodic line leaves one face of the box and re-enters at the opposite face."""
    seg = np.stack([lines[:, :-1], lines[:, 1:]], axis=2).reshape(-1, 2, 3)  # (S·(L−1), 2, 3)
    val = 0.5 * (along[:, :-1] + along[:, 1:]).reshape(-1)
    keep = np.abs(seg[:, 1] - seg[:, 0]).max(axis=1) < np.pi
    return seg[keep], val[keep]


def animate(out: dict, nu_list: list, bkm: torch.Tensor, delta: torch.Tensor, k_max: int, t_lost, path: Path) -> Path:
    t = out["t"].cpu().numpy()
    z, wmax = out["enstrophy"].cpu().numpy(), out["max_vorticity"].cpu().numpy()
    bkm = bkm.cpu().numpy()
    ts, sl, sp = out["t_save"].cpu().numpy(), out["slices"].cpu().numpy(), out["spectra"].cpu().numpy()
    dl = delta.cpu().numpy()
    names = ["Euler" if v == 0 else f"ν = {v:g}" for v in nu_list]

    clouds, vlines, valong, peak = vorticity_3d(out["w_first"])
    fig = plt.figure(figsize=(12.5, 6.8))
    grid = fig.add_gridspec(2, 3, width_ratios=[1.45, 1, 1.15])
    ax_3d = fig.add_subplot(grid[:, 0], projection="3d")
    # One colour scale for every frame and panel: the largest |ω| reached while the runs are resolved (later Euler
    # frames saturate, which is honest: that growth is a grid artefact).
    resolved = ts < t_lost if t_lost is not None else np.ones_like(ts, dtype=bool)
    w_ref = float(max(peak[resolved].max(), sl[resolved].max()))
    line_norm = plt.Normalize(0, w_ref)
    bar = fig.colorbar(plt.cm.ScalarMappable(norm=line_norm, cmap="viridis"), ax=ax_3d, shrink=0.4, pad=0.02)
    bar.set_label(r"$|\omega|$ on vortex lines (fixed scale)", fontsize=7)
    bar.ax.tick_params(labelsize=6)
    ax_e, ax_v = fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[0, 2])
    ax_t, ax_s = fig.add_subplot(grid[1, 1]), fig.add_subplot(grid[1, 2])
    ims = []
    for ax, b in ((ax_e, 0), (ax_v, 1)):
        ims.append(ax.imshow(sl[0, b].T, origin="lower", extent=[0, 2 * np.pi] * 2, cmap="magma", vmin=0, vmax=w_ref))
        ax.set_xticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
        ax.set_yticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
        ax.grid(False)
        ax.set_xlabel("$x$")
    ax_e.set_ylabel("$y$")
    titles = [ax_e.set_title("", fontsize=9), ax_v.set_title("", fontsize=9)]

    for b, name in enumerate(names):
        ax_t.semilogy(t, wmax[:, b], color=ACCENT[b], lw=1.5, label=rf"$\|\omega\|_\infty$ {name}")
        ax_t.semilogy(t, z[:, b], color=ACCENT[b], lw=1.0, ls="--")
    ax_t.plot([], [], color="#6b7280", ls="--", lw=1, label=r"enstrophy $\Omega$ (dashed)")
    if t_lost is not None:
        ax_t.axvspan(t_lost, t[-1], color="#fde68a", alpha=0.5, lw=0)
        ax_t.text(t_lost + 0.05, 0.6 * np.nanmax(wmax), "Euler\nunder-resolved", fontsize=7, va="top", color="#92400e")
    cursor = ax_t.axvline(0, color="#9ca3af", lw=1)
    ax_t.set_xlabel("$t$")
    ax_t.legend(fontsize=6.5, loc="lower right")
    bkm_text = ax_t.set_title("", fontsize=9)

    kappa = np.arange(sp.shape[-1])
    lines = [
        ax_s.semilogy(kappa[1:], sp[0, b, 1:], color=ACCENT[b], lw=1.3, label=names[b])[0] for b in range(len(names))
    ]
    ax_s.axvline(k_max, color="#9ca3af", ls="--", lw=1)
    ax_s.text(k_max + 0.3, 1e-3, "2/3 cut", fontsize=7, color="#6b7280", rotation=90)
    ax_s.set_ylim(max(sp[:, :, 1:].min(), 1e-22), 1)
    ax_s.set_xlim(0, int(np.sqrt(3) * k_max) + 1)
    ax_s.set_xlabel(r"$|k|$")
    ax_s.legend(fontsize=7, loc="upper right")
    spec_title = ax_s.set_title("", fontsize=9)

    def draw_3d(f: int) -> None:
        ax_3d.cla()
        pts, val = clouds[f]
        ax_3d.scatter(*pts.T, c=val, cmap="inferno", vmin=0, vmax=w_ref, s=4, alpha=0.35, lw=0, depthshade=False)
        seg, val = line_segments(vlines[f], valong[f])
        ax_3d.add_collection3d(Line3DCollection(seg, cmap="viridis", norm=line_norm, array=val, lw=1.3))
        for setter in (ax_3d.set_xlim, ax_3d.set_ylim, ax_3d.set_zlim):
            setter(0, 2 * np.pi)
        ax_3d.set_xticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
        ax_3d.set_yticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
        ax_3d.set_zticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
        ax_3d.set_xlabel("$x$")
        ax_3d.set_ylabel("$y$")
        ax_3d.set_zlabel("$z$")
        ax_3d.view_init(elev=22, azim=-60 + 1.5 * f)  # slow turn
        late = t_lost is not None and ts[f] >= t_lost
        ax_3d.set_title(
            r"Euler: $|\omega| \geq \frac{1}{2}|\omega|_{\max}(0)$ (dots) and vortex lines" + "\n"
            rf"$\max|\omega|$ = {peak[f]:.1f}, $t$ = {ts[f]:.2f}" + (" (under-resolved)" if late else ""),
            fontsize=9,
        )

    def update(f: int):
        draw_3d(f)
        for b in range(2):
            ims[b].set_data(sl[f, b].T)  # the same colour scale in every frame
            titles[b].set_text(rf"$\max_z|\omega|$ (peak {sl[f, b].max():.1f}), {names[b]}, $t$ = {ts[f]:.2f}")
        cursor.set_xdata([ts[f]])
        i = int(np.searchsorted(t, ts[f] - 1e-9))
        bkm_text.set_text(
            r"BKM $\int\|\omega\|_\infty dt$: " + " / ".join(f"{bkm[i, b]:.1f}" for b in range(len(names)))
        )
        for b, line in enumerate(lines):
            line.set_ydata(sp[f, b, 1:])
        d = dl[f, 0]
        if not np.isfinite(d):
            spec_title.set_text(r"energy spectrum $E(|k|)$")
        elif d > 0:
            spec_title.set_text(rf"$E(|k|)$, Euler strip $\delta = {d:.3f}$")
        else:
            spec_title.set_text(r"$E(|k|)$, Euler tail no longer decays")

    fig.tight_layout()
    return save_gif_shared_palette(fig, update, frame_indices(len(ts), len(ts)), path, fps=10)


def save_gif_shared_palette(fig, update, frames: list[int], path: Path, fps: int = 10) -> Path:
    """Like plotting.save_gif, but every frame uses ONE palette built from all frames, so a given |ω| keeps exactly
    the same colour throughout (Pillow otherwise picks a new 256-colour palette per frame and colours drift)."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)  # numbers behind the final frame (Results page)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]  # a strip of frames spanning the run sets the palette
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


def _vtk_array(values: np.ndarray, name: str, dtype: str, components: int = 1, unit: str | None = None) -> str:
    """One <DataArray> in VTK's binary XML encoding: base64 of (UInt32 byte count + zlib-compressed block)."""
    raw = np.ascontiguousarray(values, dtype={"Float32": "<f4", "Int32": "<i4"}[dtype]).tobytes()
    packed = zlib.compress(raw, 6)
    header = np.array([1, len(raw), len(raw), len(packed)], dtype="<u4").tobytes()  # 1 block, sizes
    data = base64.b64encode(header).decode() + base64.b64encode(packed).decode()
    extra = f' unit="{unit}"' if unit else ""
    return (f'<DataArray type="{dtype}" Name="{name}" NumberOfComponents="{components}" format="binary"{extra}>'
            f"{data}</DataArray>")  # fmt: skip


def write_vtp(path: Path, points: np.ndarray, cells: list | None = None, kind: str = "Polys",
              point_data: dict | None = None) -> Path:  # fmt: skip
    """Write VTK PolyData (.vtp, XML, zlib-compressed binary). points (P, 3); cells: list of index arrays (one per
    polygon / polyline / vertex), stored as Int32 connectivity + offsets under <Polys>, <Lines> or <Verts>;
    point_data: {name: (values (P,) or (P, k), unit)}."""
    cells = cells if cells is not None else [np.arange(len(points))]
    conn = np.concatenate([np.asarray(c) for c in cells]) if cells else np.zeros(0)
    offsets = np.cumsum([len(c) for c in cells]) if cells else np.zeros(0)
    counts = {k: 0 for k in ("Verts", "Lines", "Polys")}
    counts[kind] = len(cells)
    pd = "".join(
        _vtk_array(v, name, "Float32", 1 if np.ndim(v) == 1 else np.shape(v)[1], unit)
        for name, (v, unit) in (point_data or {}).items()
    )
    xml = (
        '<?xml version="1.0"?>\n<VTKFile type="PolyData" version="1.0" byte_order="LittleEndian" '
        'header_type="UInt32" compressor="vtkZLibDataCompressor"><PolyData>'
        f'<Piece NumberOfPoints="{len(points)}" NumberOfVerts="{counts["Verts"]}" NumberOfLines="{counts["Lines"]}" '
        f'NumberOfStrips="0" NumberOfPolys="{counts["Polys"]}">'
        f"<PointData>{pd}</PointData><Points>{_vtk_array(points, 'Points', 'Float32', 3)}</Points>"
        f"<{kind}>{_vtk_array(conn, 'connectivity', 'Int32')}{_vtk_array(offsets, 'offsets', 'Int32')}</{kind}>"
        "</Piece></PolyData></VTKFile>\n"
    )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(xml, encoding="utf-8")
    return path


def write_pvd(path: Path, entries: list[tuple[float, str, str]]) -> Path:
    """ParaView collection: entries (time, layer name, file relative to the .pvd); each layer is one 'part'."""
    layers = list(dict.fromkeys(name for _, name, _ in entries))
    rows = "".join(
        f'<DataSet timestep="{t:.6g}" group="" part="{layers.index(name)}" name="{name}" file="{file}"/>'
        for t, name, file in entries
    )
    path = Path(path)
    path.write_text(f'<?xml version="1.0"?>\n<VTKFile type="Collection" version="1.0" byte_order="LittleEndian">'
                    f"<Collection>{rows}</Collection></VTKFile>\n", encoding="ascii")  # fmt: skip
    return path


def read_vtp(path: Path) -> dict:
    """Parse a .vtp written by write_vtp back: counts, point coordinates and named point arrays (for the tests)."""
    import xml.etree.ElementTree as ET

    def decode(node) -> np.ndarray:
        text = node.text.strip()
        head = np.frombuffer(base64.b64decode(text[:24]), dtype="<u4")  # 4 UInt32 = 16 bytes = 24 base64 chars
        raw = zlib.decompress(base64.b64decode(text[24:]))
        assert len(raw) == head[1]
        return np.frombuffer(raw, dtype={"Float32": "<f4", "Int32": "<i4"}[node.get("type")])

    piece = ET.parse(path).getroot().find("PolyData/Piece")
    out = {k: int(piece.get(f"NumberOf{k}")) for k in ("Points", "Verts", "Lines", "Polys")}
    out["points"] = decode(piece.find("Points/DataArray")).reshape(-1, 3)
    out["arrays"] = {a.get("Name"): decode(a) for a in piece.find("PointData")}
    for kind in ("Verts", "Lines", "Polys"):
        node = piece.find(kind)
        if node is not None:
            arrays = {a.get("Name"): decode(a) for a in node}
            out["connectivity"], out["offsets"] = arrays["connectivity"], arrays["offsets"]
    return out


def export_vtk(out: dict, t_lost, out_dir: Path, n_frames: int = 20) -> Path:
    """Animated VTK of the Euler run for the web viewer and ParaView, in results/: vorticity.pvd with one layer, iso:
    the surface |ω| = ½ of the initial max |ω| (surface nets on the periodic box) at about 20 saved times up to t_lost
    (while the run is resolved). Point array `vorticity` is |ω| one grid step inside the surface along ∇|ω|, i.e. how
    strong the vortex sheet is under the iso-surface (on the surface itself |ω| is the constant level)."""
    from .tensors import surface_nets

    res = Path(out_dir) / "results"
    w = out["w_first"]  # (S, 3, N, N, N), Euler run
    ts = out["t_save"].cpu().numpy()
    n = w.shape[-1]
    h = 2 * np.pi / n
    mag = w.norm(dim=1)  # (S, N, N, N)
    level = 0.5 * float(mag[0].max())
    resolved = np.nonzero(ts < t_lost)[0] if t_lost is not None else np.arange(len(ts))
    picks = resolved[frame_indices(len(resolved), n_frames)]
    entries = []
    for j, s in enumerate(picks):
        verts, faces = surface_nets(mag[s], level, periodic=True)  # grid units
        if len(faces) == 0:
            continue
        x = verts.to(w.dtype)[None] * h  # (1, V, 3) physical positions
        m3 = mag[s][None, None].expand(1, 3, n, n, n)
        grad = torch.stack([(torch.roll(mag[s], -1, a) - torch.roll(mag[s], 1, a)) / (2 * h) for a in range(3)])[None]
        g = sample_periodic(grad, x % (2 * np.pi))
        inward = x + h * g / g.norm(dim=-1, keepdim=True).clamp_min(1e-12)  # towards stronger |ω|
        value = sample_periodic(m3.contiguous(), inward % (2 * np.pi))[0, :, 0].cpu().numpy()
        name = f"iso_{j:03d}.vtp"
        write_vtp(res / name, x[0].cpu().numpy(), list(faces.cpu().numpy()), "Polys",
                  {"vorticity": (value, "a.u.")})  # fmt: skip
        entries.append((float(ts[s]), "iso", name))
    return write_pvd(res / "vorticity.pvd", entries)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n': 32,
    'nu': 0.015,
    't_max': 6.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.navier_stokes_blowup")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.navier_stokes_blowup.main")
problem_dir = Path("problems/fluids/navier_stokes_blowup")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
